In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

# Confound check: is the linear-probe accuracy driven by text length / lexical content?

The probes in `linear_probe.ipynb` hit ~95-100% accuracy at every layer,
including layer 0 (raw embeddings). That's suspicious: `neutral` examples
are one short factual sentence; persuasion-technique examples are full
paragraphs with citations, named experts, and statistics. This notebook
checks how much of that separability is explained by trivial surface
features (length, bag-of-words) rather than activations reflecting a
deeper "persuasion" representation.

**1. Inspect**: `data/activations/run_1/*/metadata.json` does *not* contain
the prompt/response text -- only `ids`, `variant`, `source_filename`,
`model_id`, `num_examples`. The actual text lives in
`data/responses/persuasion_dataset_with_user.json`, one row per `id`, with
one field per variant (`neutral`, `evidence_based_persuasion`, ... and the
`_tl` Filipino counterparts). `collect_activations.py` confirms
`assistant_content = row[variant]` -- i.e. each activation folder's
`mean_assistant_token.safetensors` is the mean hidden state over exactly
that field's text, fed through the model as the assistant turn. So mapping
an activation row back to text is: `row[variant]` for English, or
`row[variant + "_tl"]` for Filipino, joined on `id`.

Confirmed confound, just from row 0: `neutral` = 7 words, `evidence_based_persuasion`
= 129 words (en); `neutral_tl` = 13 words, `evidence_based_persuasion_tl` = 133
words (tl).

In [2]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from safetensors import safe_open
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import pointbiserialr

pd.set_option("future.infer_string", False)

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RUN_DIR = PROJECT_ROOT / "data" / "activations" / "run_1"
RESPONSES_PATH = PROJECT_ROOT / "data" / "responses" / "persuasion_dataset_with_user.json"

with open(RESPONSES_PATH, "r", encoding="utf-8") as f:
    _dataset = json.load(f)
ID2ROW = {row["id"]: row for row in _dataset}
print(f"{len(ID2ROW)} prompts loaded from {RESPONSES_PATH.name}")

200 prompts loaded from persuasion_dataset_with_user.json


In [3]:
TECHNIQUE_SHORT = {
    "authority_endorsement_persuasion": "authority",
    "evidence_based_persuasion": "evidence",
    "expert_endorsement_persuasion": "expert",
    "logical_appeal_persuasion": "logical",
    "misrepresentation_persuasion": "misrep",
}

LANGUAGES = {
    "en": {"neutral": "neutral", "techniques": list(TECHNIQUE_SHORT.keys())},
    "tl": {"neutral": "neutral_tl", "techniques": [t + "_tl" for t in TECHNIQUE_SHORT.keys()]},
}

ID_FILTER_IN_DOMAIN = "everyday"
TEST_SIZE = 0.2
SPLIT_SEED = 42
PROBE_LAYERS = [0, 12]


def load_activation_samples(folder: str, id_filter: str):
    """([n_samples, n_layers, hidden], ids) for one variant folder, filtered to ids containing id_filter."""
    path = RUN_DIR / folder / "mean_assistant_token.safetensors"
    with safe_open(str(path), framework="pt") as f:
        tensor = f.get_tensor("activations")
        ids = json.loads(f.metadata()["ids"])
    keep = [i for i, sample_id in enumerate(ids) if id_filter in sample_id]
    return tensor[keep].numpy(), [ids[i] for i in keep]


def text_for(folder: str, sample_id: str) -> str:
    """The exact text collect_activations.py fed in as the assistant turn for this (folder, id)."""
    return ID2ROW[sample_id][folder]


def text_features(text: str) -> tuple[int, int, int]:
    char_len = len(text)
    word_len = len(text.split())
    sentence_count = len([s for s in re.split(r"[.!?]+", text) if s.strip()])
    return char_len, word_len, sentence_count

## Build (activations, text, labels) per probe

One probe per (language, technique-group): `combined` pools all 5
techniques as the positive class vs. that language's `neutral`; the other
5 use a single technique each. Same `everyday`-filtered, 80/20 stratified
split used by the existing activation probes (`test_size=0.2,
random_state=42`), so every baseline below is evaluated on the *exact same*
held-out rows as the activation probe.

In [4]:
def build_probe_data(lang: str, technique_folders: list[str]):
    neutral_folder = LANGUAGES[lang]["neutral"]
    neutral_acts, neutral_ids = load_activation_samples(neutral_folder, ID_FILTER_IN_DOMAIN)
    technique_acts, technique_ids = [], []
    for folder in technique_folders:
        acts, ids = load_activation_samples(folder, ID_FILTER_IN_DOMAIN)
        technique_acts.append(acts)
        technique_ids.extend((folder, i) for i in ids)

    X = np.concatenate([neutral_acts] + technique_acts, axis=0)
    y = np.concatenate([np.zeros(len(neutral_ids)), np.ones(len(technique_ids))])

    texts = [text_for(neutral_folder, i) for i in neutral_ids] + [
        text_for(folder, i) for folder, i in technique_ids
    ]
    return X, y, texts


def fit_mm_direction(X_layer: np.ndarray, y_layer: np.ndarray):
    pos_mean = X_layer[y_layer == 1].mean(axis=0)
    neg_mean = X_layer[y_layer == 0].mean(axis=0)
    direction = pos_mean - neg_mean
    threshold = ((pos_mean + neg_mean) / 2) @ direction
    return direction, threshold


def mm_predict(X_layer: np.ndarray, direction: np.ndarray, threshold: float) -> np.ndarray:
    return (X_layer @ direction > threshold).astype(int)

## Per-probe analysis: length stats, correlation, and the 4 comparisons

In [5]:
def analyze_probe(lang: str, technique_label: str, technique_folders: list[str]) -> dict:
    X, y, texts = build_probe_data(lang, technique_folders)
    feats = np.array([text_features(t) for t in texts])  # [n, 3] = char, word, sentence
    char_len, word_len, sentence_count = feats[:, 0], feats[:, 1], feats[:, 2]

    train_idx, test_idx = train_test_split(
        np.arange(len(y)), test_size=TEST_SIZE, stratify=y, random_state=SPLIT_SEED
    )
    y_train, y_test = y[train_idx], y[test_idx]

    # --- length stats per class ---
    len_mean_neutral = word_len[y == 0].mean()
    len_std_neutral = word_len[y == 0].std()
    len_mean_pos = word_len[y == 1].mean()
    len_std_pos = word_len[y == 1].std()
    corr, corr_p = pointbiserialr(y, word_len)

    # --- length-only baseline ---
    scaler = StandardScaler().fit(feats[train_idx])
    len_clf = LogisticRegression(max_iter=1000).fit(scaler.transform(feats[train_idx]), y_train)
    len_pred = len_clf.predict(scaler.transform(feats[test_idx]))
    len_acc = accuracy_score(y_test, len_pred)
    len_bal_acc = balanced_accuracy_score(y_test, len_pred)

    # --- bag-of-words baseline ---
    vec = TfidfVectorizer(max_features=2000)
    Xtr_bow = vec.fit_transform([texts[i] for i in train_idx])
    Xte_bow = vec.transform([texts[i] for i in test_idx])
    bow_clf = LogisticRegression(max_iter=1000).fit(Xtr_bow, y_train)
    bow_pred = bow_clf.predict(Xte_bow)
    bow_acc = accuracy_score(y_test, bow_pred)
    bow_bal_acc = balanced_accuracy_score(y_test, bow_pred)

    # --- activation probes at layer 0 and layer 12, same split ---
    probe_results = {}
    for layer in PROBE_LAYERS:
        direction, threshold = fit_mm_direction(X[train_idx, layer, :], y_train)
        pred = mm_predict(X[test_idx, layer, :], direction, threshold)
        probe_results[layer] = (accuracy_score(y_test, pred), balanced_accuracy_score(y_test, pred))

    return {
        "lang": lang,
        "technique": technique_label,
        "n": len(y),
        "len_mean_neutral": len_mean_neutral,
        "len_std_neutral": len_std_neutral,
        "len_mean_persuasive": len_mean_pos,
        "len_std_persuasive": len_std_pos,
        "length_label_corr": corr,
        "length_label_corr_p": corr_p,
        "length_only_acc": len_acc,
        "length_only_bal_acc": len_bal_acc,
        "bow_acc": bow_acc,
        "bow_bal_acc": bow_bal_acc,
        "layer0_acc": probe_results[0][0],
        "layer0_bal_acc": probe_results[0][1],
        "layer12_acc": probe_results[12][0],
        "layer12_bal_acc": probe_results[12][1],
    }


def run_language(lang: str) -> pd.DataFrame:
    rows = []
    techniques = LANGUAGES[lang]["techniques"]
    rows.append(analyze_probe(lang, "combined", techniques))
    for full_name, short in TECHNIQUE_SHORT.items():
        folder = full_name + ("_tl" if lang == "tl" else "")
        rows.append(analyze_probe(lang, short, [folder]))
    return pd.DataFrame(rows)

## English

In [6]:
results_en = run_language("en")
results_en

,lang,technique,n,len_mean_neutral,len_std_neutral,len_mean_persuasive,len_std_persuasive,length_label_corr,length_label_corr_p,length_only_acc,length_only_bal_acc,bow_acc,bow_bal_acc,layer0_acc,layer0_bal_acc,layer12_acc,layer12_bal_acc
0,en,combined,600,8.3,1.75784,93.918,19.917612,0.868674,1.412361e-184,1.0,1.0,0.858333,0.575,1.0,1.0,1.0,1.0
1,en,authority,200,8.3,1.75784,84.450,8.862703,0.986212,1.184592e-156,1.0,1.0,1.000000,1.000,1.0,1.0,1.0,1.0
2,en,evidence,200,8.3,1.75784,125.830,13.638222,0.986586,7.952826e-158,1.0,1.0,1.000000,1.000,1.0,1.0,1.0,1.0
3,en,expert,200,8.3,1.75784,90.800,10.291744,0.984359,2.867180e-151,1.0,1.0,1.000000,1.000,1.0,1.0,1.0,1.0
4,en,logical,200,8.3,1.75784,90.510,10.714005,0.983002,1.012974e-147,1.0,1.0,1.000000,1.000,1.0,1.0,1.0,1.0
5,en,misrep,200,8.3,1.75784,78.000,10.749884,0.976437,8.076432e-134,1.0,1.0,1.000000,1.000,1.0,1.0,1.0,1.0


## Tagalog

In [7]:
results_tl = run_language("tl")
results_tl

,lang,technique,n,len_mean_neutral,len_std_neutral,len_mean_persuasive,len_std_persuasive,length_label_corr,length_label_corr_p,length_only_acc,length_only_bal_acc,bow_acc,bow_bal_acc,layer0_acc,layer0_bal_acc,layer12_acc,layer12_bal_acc
0,tl,combined,600,12.19,2.415347,108.934,19.488192,0.896503,1.356276e-213,1.0,1.0,0.925,0.775,0.983333,0.95,1.0,1.0
1,tl,authority,200,12.19,2.415347,91.770,9.234560,0.985916,9.545523e-156,1.0,1.0,1.000,1.000,0.950000,0.95,1.0,1.0
2,tl,evidence,200,12.19,2.415347,138.750,13.972383,0.987679,1.858315e-161,1.0,1.0,1.000,1.000,0.950000,0.95,1.0,1.0
3,tl,expert,200,12.19,2.415347,100.890,11.198120,0.983726,1.406615e-149,1.0,1.0,1.000,1.000,0.950000,0.95,1.0,1.0
4,tl,logical,200,12.19,2.415347,110.000,11.227644,0.986492,1.576774e-157,1.0,1.0,1.000,1.000,0.950000,0.95,1.0,1.0
5,tl,misrep,200,12.19,2.415347,103.260,9.238636,0.989183,5.026134e-167,1.0,1.0,1.000,1.000,0.950000,0.95,1.0,1.0


## Summary table

`[comparison, length-only accuracy, bag-of-words accuracy, layer-0 probe
accuracy, layer-12 probe accuracy]`, both accuracy and balanced accuracy,
for every probe in both languages.

In [8]:
def summary_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df[["lang", "technique"]].copy()
    out["comparison"] = out["lang"] + " neutral vs. " + out["technique"]
    out["length_only_acc"] = df["length_only_acc"].round(3)
    out["length_only_bal_acc"] = df["length_only_bal_acc"].round(3)
    out["bow_acc"] = df["bow_acc"].round(3)
    out["bow_bal_acc"] = df["bow_bal_acc"].round(3)
    out["layer0_acc"] = df["layer0_acc"].round(3)
    out["layer0_bal_acc"] = df["layer0_bal_acc"].round(3)
    out["layer12_acc"] = df["layer12_acc"].round(3)
    out["layer12_bal_acc"] = df["layer12_bal_acc"].round(3)
    return out[[
        "comparison", "length_only_acc", "length_only_bal_acc",
        "bow_acc", "bow_bal_acc", "layer0_acc", "layer0_bal_acc",
        "layer12_acc", "layer12_bal_acc",
    ]]


summary_df = pd.concat([summary_table(results_en), summary_table(results_tl)], ignore_index=True)
summary_df

,comparison,length_only_acc,length_only_bal_acc,bow_acc,bow_bal_acc,layer0_acc,layer0_bal_acc,layer12_acc,layer12_bal_acc
0,en neutral vs. combined,1.0,1.0,0.858,0.575,1.000,1.00,1.0,1.0
1,en neutral vs. authority,1.0,1.0,1.000,1.000,1.000,1.00,1.0,1.0
2,en neutral vs. evidence,1.0,1.0,1.000,1.000,1.000,1.00,1.0,1.0
3,en neutral vs. expert,1.0,1.0,1.000,1.000,1.000,1.00,1.0,1.0
4,en neutral vs. logical,1.0,1.0,1.000,1.000,1.000,1.00,1.0,1.0
5,en neutral vs. misrep,1.0,1.0,1.000,1.000,1.000,1.00,1.0,1.0
6,tl neutral vs. combined,1.0,1.0,0.925,0.775,0.983,0.95,1.0,1.0
7,tl neutral vs. authority,1.0,1.0,1.000,1.000,0.950,0.95,1.0,1.0
8,tl neutral vs. evidence,1.0,1.0,1.000,1.000,0.950,0.95,1.0,1.0
9,tl neutral vs. expert,1.0,1.0,1.000,1.000,0.950,0.95,1.0,1.0


## Length vs. label correlation, and length vs. layer-12 projection

In [9]:
print("Point-biserial correlation: word length vs. label (1=persuasive)")
print(pd.concat([results_en, results_tl], ignore_index=True)[["lang", "technique", "length_label_corr", "length_label_corr_p"]])

Point-biserial correlation: word length vs. label (1=persuasive)
   lang  technique  length_label_corr  length_label_corr_p
0    en   combined           0.868674        1.412361e-184
1    en  authority           0.986212        1.184592e-156
2    en   evidence           0.986586        7.952826e-158
3    en     expert           0.984359        2.867180e-151
4    en    logical           0.983002        1.012974e-147
5    en     misrep           0.976437        8.076432e-134
6    tl   combined           0.896503        1.356276e-213
7    tl  authority           0.985916        9.545523e-156
8    tl   evidence           0.987679        1.858315e-161
9    tl     expert           0.983726        1.406615e-149
10   tl    logical           0.986492        1.576774e-157
11   tl     misrep           0.989183        5.026134e-167


### Optional: length vs. projection onto the layer-12 diff-of-means direction

If the activation direction is just re-deriving "longer text," the
projection should correlate with word length *within* each class, not just
separate the two classes (which would just restate that the classes differ
in length).

In [10]:
def projection_length_corr(lang: str, technique_label: str, technique_folders: list[str], layer: int = 12):
    X, y, texts = build_probe_data(lang, technique_folders)
    word_len = np.array([len(t.split()) for t in texts])

    direction, _ = fit_mm_direction(X[:, layer, :], y)
    projection = X[:, layer, :] @ direction

    corr_all = np.corrcoef(word_len, projection)[0, 1]
    corr_neutral = np.corrcoef(word_len[y == 0], projection[y == 0])[0, 1]
    corr_pos = np.corrcoef(word_len[y == 1], projection[y == 1])[0, 1]
    return {
        "lang": lang, "technique": technique_label,
        "corr_length_vs_projection_overall": corr_all,
        "corr_length_vs_projection_neutral_only": corr_neutral,
        "corr_length_vs_projection_persuasive_only": corr_pos,
    }


proj_rows = []
for lang in ["en", "tl"]:
    techniques = LANGUAGES[lang]["techniques"]
    proj_rows.append(projection_length_corr(lang, "combined", techniques))
    for full_name, short in TECHNIQUE_SHORT.items():
        folder = full_name + ("_tl" if lang == "tl" else "")
        proj_rows.append(projection_length_corr(lang, short, [folder]))

pd.DataFrame(proj_rows)

,lang,technique,corr_length_vs_projection_overall,corr_length_vs_projection_neutral_only,corr_length_vs_projection_persuasive_only
0,en,combined,0.878393,0.509107,0.339953
1,en,authority,0.978178,0.533684,0.451488
2,en,evidence,0.976117,0.551917,0.288670
3,en,expert,0.975833,0.460055,0.265497
4,en,logical,0.973597,0.543250,0.259273
5,en,misrep,0.967868,0.437504,0.269699
6,tl,combined,0.906796,0.509827,0.391774
7,tl,authority,0.973551,0.482949,0.322053
8,tl,evidence,0.977099,0.508685,0.304762
9,tl,expert,0.976121,0.505531,0.353386
